# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [4]:
# Write your code below.

from dotenv import load_dotenv
load_dotenv()

True

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import os
from glob import glob

# Write your code below

price_data_dir = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(price_data_dir, "**", "*.parquet"), recursive=True)

print("PRICE_DATA dir:", price_data_dir)
print("Parquet files found:", parquet_files)

PRICE_DATA dir: ../../05_src/data/prices/
Parquet files found: ['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.1.pa

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [7]:
# Write your code below.

import dask.dataframe as dd

ddf = dd.read_parquet(parquet_files)
ddf['Date'] = dd.to_datetime(ddf['Date'])
ddf = ddf.set_index('Date')

def add_features(df):
    df = df.sort_index()
    df['Close_lag_1'] = df['Close'].shift(1)
    df['Adj_Close_lag_1'] = df['Adj_Close'].shift(1)
    df['returns'] = (df['Close'] / df['Close_lag_1']) - 1
    df['hi_lo_range'] = df['High'] - df['Low']
    return df

dd_feat = ddf.groupby('ticker').apply(
    add_features,
    meta={
        'Ticker': 'object',
        'Close': 'float64',
        'Adj_Close': 'float64',
        'High': 'float64',
        'Low': 'float64',
        'Close_lag_1': 'float64',
        'Adj_Close_lag_1': 'float64',
        'returns': 'float64',
        'hi_lo_range': 'float64'
    }
)

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [11]:
# Write your code below.
# Convert to pandas
feat_df = dd_feat.compute()

# Add moving average of returns
feat_df['returns_ma_10'] = feat_df.groupby('ticker')['returns'].rolling(10).mean().reset_index(level=0, drop=True)

feat_df.head()

KeyboardInterrupt: 

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
No, we could have done it in Dask, but it may have been less flexible than Pandas.

+ Would it have been better to do it in Dask? Why?
It may have been faster.
(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.